# باب 7: چیٹ ایپلیکیشنز بنانا  
## اوپن اے آئی API جلد آغاز  

یہ نوٹ بک [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) سے ماخوذ ہے جس میں نوٹ بکس شامل ہیں جو [Azure OpenAI](notebook-azure-openai.ipynb) سروسز تک رسائی دیتی ہیں۔  

Python OpenAI API کچھ تبدیلیوں کے ساتھ Azure OpenAI ماڈلز کے ساتھ بھی کام کرتا ہے۔ یہاں مزید جانیں کہ فرق کیا ہیں: [Python کے ساتھ OpenAI اور Azure OpenAI اینڈ پوائنٹس کے درمیان کیسے سوئچ کریں](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)  


# جائزہ  
"بڑے زبان کے ماڈلز ایسے فعل ہوتے ہیں جو متن کو متن میں نقش کرتے ہیں۔ جب ایک ان پٹ ٹیکسٹ سٹرنگ دی جاتی ہے، تو ایک بڑا زبان ماڈل اس متن کی پیش گوئی کرنے کی کوشش کرتا ہے جو اگلے آئے گا"(1)۔ یہ "فوری آغاز" نوٹ بک صارفین کو اعلی سطح کے LLM تصورات، AML کے ساتھ شروع کرنے کے لیے بنیادی پیکج کی ضروریات، پرامپٹ ڈیزائن کا ہلکا تعارف، اور مختلف استعمال کے واقعات کی کئی چھوٹی مثالوں سے متعارف کرائے گی۔ 


## فہرست مضامین  

[جائزہ](#overview)  
[اوپن اے آئی سروس استعمال کرنے کا طریقہ](#how-to-use-openai-service)  
[1. اپنی اوپن اے آئی سروس بنانا](#1.-creating-your-openai-service)  
[2. تنصیب](#2.-installation)    
[3. اسناد](#3.-credentials)  

[استعمال کے کیسز](#use-cases)    
[1. متن کا خلاصہ بنانا](#1.-summarize-text)  
[2. متن کی درجہ بندی](#2.-classify-text)  
[3. نئے مصنوعات کے نام تیار کرنا](#3.-generate-new-product-names)  
[4. درجہ بندی کرنے والے کو بہتر بنانا](#4.fine-tune-a-classifier)  

[حوالہ جات](#references)


### اپنا پہلا پرامپٹ بنائیں  
یہ مختصر مشق ایک بنیادی تعارف فراہم کرے گی کہ کیسے اوپن اے آئی ماڈل کو ایک سادہ کام "خلاصہ" کے لیے پرامپٹ بھیجا جائے۔  


**اقدامات**:  
1. اپنے پائتھن ماحول میں OpenAI لائبریری انسٹال کریں  
2. معیاری ہیلپر لائبریریز لوڈ کریں اور اپنے بنائے ہوئے OpenAI سروس کے لئے OpenAI سیکیورٹی اسناد سیٹ کریں  
3. اپنے کام کے لیے ایک ماڈل منتخب کریں  
4. ماڈل کے لئے ایک سادہ پرامپٹ بنائیں  
5. اپنے درخواست کو ماڈل API پر بھیجیں!  


### 1. اوپن اے آئی انسٹال کریں


In [ ]:
%pip install openai python-dotenv

### 2. معاون لائبریریاں درآمد کریں اور اسناد کا مظاہرہ کریں


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. درست ماڈل تلاش کرنا  
GPT-4o اور GPT-4o mini جیسے ماڈل قدرتی زبان کو سمجھ سکتے ہیں اور تخلیق کر سکتے ہیں، اور OpenAI پلیٹ فارم پر مختلف کاموں کے لیے مختلف طاقت اور رفتار کی سطحوں کے ساتھ دستیاب ہیں۔


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. پرامپٹ ڈیزائن  

"بڑے زبان کے ماڈلز کی جادو یہ ہے کہ جب انہیں بہت زیادہ مقدار میں متن پر پیش گوئی کی غلطی کو کم سے کم کرنے کی تربیت دی جاتی ہے، تو ماڈلز ایسے تصورات سیکھ لیتے ہیں جو ان پیش گوئیوں کے لیے مفید ہوتے ہیں۔ مثلاً، وہ درج ذیل تصورات سیکھتے ہیں"(1):

* ہجے کیسے کیے جاتے ہیں
* گرامر کیسے کام کرتی ہے
* پیرایہ کیسے بنایا جاتا ہے
* سوالات کیسے جواب دیے جاتے ہیں
* گفتگو کیسے کی جاتی ہے
* مختلف زبانوں میں کیسے لکھا جاتا ہے
* کوڈ کیسے لکھا جاتا ہے
* وغیرہ۔

#### بڑے زبان کے ماڈل کو کیسے کنٹرول کیا جائے  
"بڑے زبان کے ماڈل میں تمام ان پٹ میں سب سے زیادہ اثرانداز ہونے والا عنصر ٹیکسٹ پرامپٹ ہے(1)۔

بڑے زبان کے ماڈلز کو چند طریقوں سے آؤٹ پٹ پیدا کرنے کے لیے پرامپٹ کیا جا سکتا ہے:

ہدایت: ماڈل کو بتائیں کہ آپ کیا چاہتے ہیں
تکمیل: ماڈل کو اس کی شروعات مکمل کرنے پر اکسانا جو آپ چاہتے ہیں
مظاہرہ: ماڈل کو دکھائیں کہ آپ کیا چاہتے ہیں، یا تو:
پرامپٹ میں چند مثالیں
بارہاں یا ہزاروں مثالیں فائن ٹوننگ تربیتی ڈیٹا سیٹ میں"



#### پرامپٹس بنانے کے تین بنیادی اصول ہیں:

**دکھائیں اور بتائیں**۔ واضح کریں کہ آپ کیا چاہتے ہیں چاہے ہدایات کے ذریعے، مثالوں کے ذریعے، یا دونوں کا امتزاج ہو۔ اگر آپ چاہتے ہیں کہ ماڈل اشیاء کی فہرست کو حروف تہجی کی ترتیب میں درجہ بند کرے یا پیراگراف کو جذبات کی بنیاد پر درجہ بندی کرے، تو اسے دکھائیں کہ یہی آپ کی خواہش ہے۔

**معیاری ڈیٹا فراہم کریں**۔ اگر آپ کوئی کلاسیفائر بنا رہے ہیں یا ماڈل کو کسی نمونے کی پیروی کروا رہے ہیں تو اس بات کو یقینی بنائیں کہ مثالیں کافی ہوں۔ اپنی مثالوں کو پروف ریڈ کرنا یقینی بنائیں — ماڈل عام طور پر بنیادی ہجے کی غلطیاں دیکھ سکتا ہے اور جواب دے سکتا ہے، لیکن ہو سکتا ہے کہ اسے یہ جان بوجھ کر کیا گیا سمجھ کر جواب پر اثر پڑے۔

**اپنی ترتیبات چیک کریں۔** درجہ حرارت اور top_p سیٹنگز کنٹرول کرتی ہیں کہ ماڈل جواب دینے میں کتنا متعین ہے۔ اگر آپ ایسے جواب چاہتے ہیں جہاں صرف ایک درست جواب ہو، تو آپ کو انہیں کم کرنا چاہیے۔ اگر آپ متنوع جوابات چاہتے ہیں تو آپ انہیں زیادہ کر سکتے ہیں۔ ان سیٹنگز کے ساتھ سب سے بڑی غلطی یہ ہے کہ لوگ انہیں "چالاکی" یا "تخلیقی صلاحیت" کے کنٹرول سمجھ بیٹھتے ہیں۔


ماخذ: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. جمع کروائیں!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### ایک ہی کال کو دہرائیں، نتائج کا موازنہ کیسے کیا جائے؟


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## متن کا خلاصہ کریں  
#### چیلنج  
ایک متن کے حصے کے آخر میں 'tl;dr:' شامل کرکے متن کا خلاصہ کریں۔ دھیان دیں کہ ماڈل اضافی ہدایات کے بغیر متعدد کام انجام دینے کا طریقہ کس طرح سمجھتا ہے۔ آپ ماڈل کے رویے میں تبدیلی اور حاصل شدہ خلاصہ کو حسب ضرورت بنانے کے لیے tl;dr سے زیادہ وضاحتی پرامپٹس کے ساتھ تجربہ کر سکتے ہیں(3).  

حالیہ کام نے بڑے متن کے مجموعے پر پری ٹریننگ اور اس کے بعد ایک مخصوص کام پر فائن ٹیوننگ کے ذریعے NLP کے کئی کاموں اور بینچ مارکس پر نمایاں فوائد دکھائے ہیں۔ اگرچہ عام طور پر فن تعمیر میں کام سے آزاد ہوتا ہے، اس طریقہ کار کو اب بھی ہزاروں یا لاکھوں مخصوص کام کے فائن ٹیوننگ ڈیٹا سیٹس کی ضرورت ہوتی ہے۔ اس کے برعکس، انسان عموماً چند مثالوں یا آسان ہدایات سے نیا زبان کا کام انجام دے سکتے ہیں - ایسا کچھ جو موجودہ NLP سسٹمز اب بھی بڑی حد تک کرنے میں دشواری محسوس کرتے ہیں۔ یہاں ہم دکھاتے ہیں کہ زبان کے ماڈلز کو اسکیل کرنے سے کام سے آزاد، چند شوٹ کارکردگی میں بہت بہتری آتی ہے، جو بعض اوقات پچھلے اعلیٰ ترین فائن ٹیوننگ طریقوں کے مقابلے کے قابل ہوتی ہے۔  



خلاصہ  


# متعدد استعمال کے کیسز کے لیے مشقیں  
1. متن کا خلاصہ بنائیں  
2. متن کو درجہ بندی کریں  
3. نئے مصنوعات کے نام تیار کریں


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## متن کی درجہ بندی کریں  
#### چیلنج  
آئٹمز کو اُن زمروں میں تقسیم کریں جو اندازے کے وقت فراہم کیے گئے ہوں۔ درج ذیل مثال میں، ہم زمروں اور متن دونوں کو درجہ بندی کے لیے پرامپٹ میں فراہم کرتے ہیں(*playground_reference). 

کسٹمر کی استفسار: ہیلو، میرے لیپ ٹاپ کے کی بورڈ میں سے ایک کی جگہ حال ہی میں ٹوٹ گئی ہے اور مجھے اس کی جگہ نیا چاہیے:

درجہ بند کیا گیا زمرہ:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## نئے مصنوعات کے نام پیدا کریں
#### چیلنج
نمونے کے الفاظ سے مصنوعات کے نام تخلیق کریں۔ یہاں ہم پرامپٹ میں اس مصنوعات کے بارے میں معلومات شامل کرتے ہیں جس کے نام ہم پیدا کرنے جا رہے ہیں۔ ہم ایک مشابہ مثال بھی فراہم کرتے ہیں تاکہ وہ نمونہ دکھایا جا سکے جو ہم حاصل کرنا چاہتے ہیں۔ ہم نے درجہ حرارت کی قدر بھی زیادہ مقرر کی ہے تاکہ بے ترتیبی میں اضافہ ہو اور مزید جدید جوابات حاصل ہوں۔

مصنوعات کی وضاحت: ایک گھر کا ملک شیک بنانے والا
بیج الفاظ: تیز، صحت بخش، کمپیکٹ۔
مصنوعات کے نام: HomeShaker, Fit Shaker, QuickShake, Shake Maker

مصنوعات کی وضاحت: جوتے کا ایک جوڑا جو کسی بھی قدم کے سائز میں فٹ ہو سکتا ہے۔
بیج الفاظ: قابل مطابقت، فٹ، اومنی فٹ۔


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# حوالہ جات  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [متن کی درجہ بندی کے لیے GPT ماڈلز کی فائن ٹیوننگ کے بہترین طریقے](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# مزید مدد کے لیے  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# معاونین
* لوئس لی  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
